In [1]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from transformers import AutoModelForCTC, Wav2Vec2Processor
import argparse
import json
from tqdm import tqdm
import os
from textgrid import TextGrid, IntervalTier,PointTier
from praatio import textgrid

import editdistance
from VAD_chunk import *


############################################
# MAPPINGS
############################################

ref_mapping = {
    "a": "a",
    "b": "b",
    "c": "k",
    "d": "d",
    "Z": "ʒ",
    "e": "e",
    "f": "f",
    "i": "i",
    "j": "j",
    "k": "k",
    "l": "l",
    "m": "mʲ",
    "n": "n",
    "o": "o",
    "p": "p",
    "s": "s",
    "t": "t",
    "u": "u",
    "v": "v",
    "w": "w",
    "y": "y",
    "z": "z",
    "2": "ø",
    "9": "œ",
    "N": "ŋ",
    "@": "ə",
    "E": "ɛ",
    "O": "ɔ",
    "R": "ʁ",
    "r": "ʁ",
    "S": "ʃ",
    "J": "ɲ",
    "H": "ɥ",
    "g": "ɡ",
    "g":"ɡ",

    # Nasals
    "a~": "ɑ̃",
    "o~": "ɔ̃",
    "e~": "ɛ̃",
    "9~": "ɛ̃",
}
hyp_mapping = {
    "ɑ": "a",
    "ɒ": "ɔ",
    "ɟ": "ɡ",
    "ɣ": "ʁ",
    "ɹ": "ʁ",
    "ɾ": "ʁ",
    "ʎ": "l",
    "mʲ": "m",
}

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_reference_alignments(textgrid_path):

    tg = TextGrid()
    tg.read(textgrid_path)

    ref_alignments = []

    # assuming tier name is "phones"
    tier = tg.getFirst("phone")

    for interval in tier.intervals:

        phoneme = interval.mark.strip()

        if phoneme == "" or phoneme in ["sil", "sp", "spn"]:
            continue

        midpoint = (interval.minTime + interval.maxTime) / 2

        ref_alignments.append({
            "phoneme": phoneme,
            "time_sec": midpoint
        })

    return ref_alignments


In [3]:


import difflib

def match_alignments(ref_alignments, pred_alignments):

    # Normalisation phonème par phonème
    ref_seq = [
        normalize_reference(p["phoneme"])
        for p in ref_alignments
    ]

    pred_seq = [
        normalize_hypothesis(p["phoneme"])
        for p in pred_alignments
    ]

    matcher = difflib.SequenceMatcher(None, ref_seq, pred_seq)

    boundary_errors = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":
            for r, p in zip(range(i1, i2), range(j1, j2)):
                error = abs(
                    ref_alignments[r]["time_sec"] -
                    pred_alignments[p]["time_sec"]
                )
                boundary_errors.append(error)

    return boundary_errors

############################################
# NORMALIZATION FUNCTIONS
############################################

def normalize_reference(ref_string):

    # Replace SAMPA → IPA (longest first)
    for k in sorted(ref_mapping.keys(), key=len, reverse=True):
        ref_string = ref_string.replace(k, ref_mapping[k])

    # Remove noise symbols
    for noise in ["_", "sil", "spn", "%", "?", "0", "="]:
        ref_string = ref_string.replace(noise, "")

    # Remove ASCII tilde (if any remains)
    ref_string = ref_string.replace("~", "")

    # Remove spaces
    ref_string = ref_string.replace(" ", "")

    return ref_string


def normalize_hypothesis(decoded_string):

    hyp = decoded_string.replace(" ", "")

    # Replace variants
    for k in sorted(hyp_mapping, key=len, reverse=True):
        hyp = hyp.replace(k, hyp_mapping[k])

    return hyp


############################################
# PER FUNCTION (CHARACTER-LEVEL)
############################################

def compute_per(ref_string, hyp_string):

    ref_tokens = list(ref_string)
    hyp_tokens = list(hyp_string)

    if len(ref_tokens) == 0:
        return 0, 0

    distance = editdistance.eval(ref_tokens, hyp_tokens)

    return distance, len(ref_tokens)


############################################
# PHONEME + ALIGNMENT
############################################

def get_phoneme_alignments(model, processor, audio_path):

    audio, sr = sf.read(audio_path)

    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

    wav = torch.from_numpy(audio)

    chunks = vad_chunk_with_timestamps(wav)

    device = next(model.parameters()).device
    frame_duration = model.config.inputs_to_logits_ratio / 16000

    all_alignments = []
    full_phoneme_string = ""

    for chunk in chunks:

        start_sample = int(chunk["start"] * 16000)
        end_sample = int(chunk["end"] * 16000)

        chunk_tensor = wav[start_sample:end_sample]

        inputs = processor(
            chunk_tensor.numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)[0]

        decoded = processor.batch_decode(predicted_ids.unsqueeze(0))[0]

        full_phoneme_string += decoded

        prev_id = None

        for frame_idx, token_id in enumerate(predicted_ids.tolist()):

            if token_id != processor.tokenizer.pad_token_id and token_id != prev_id:
                phoneme = processor.decode([token_id]).strip()

                local_time = frame_idx * frame_duration
                absolute_time = (start_sample / 16000) + local_time

                all_alignments.append({
                    "phoneme": phoneme,
                    "time_sec": absolute_time
                })

            prev_id = token_id

    return full_phoneme_string.strip(), all_alignments



In [4]:
MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cuda"
model = model.to(device)
model.eval()

/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Pl

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [10]:
textgrid_dir = "/vol/corpora/Rhapsodie/TextGrids-fev2013"
csv_path = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/rhap_phonemes_ref.csv"
audio_dir = "/vol/corpora/Rhapsodie/wav16k_corrected"
df = pd.read_csv(csv_path)
json_results = []       # for phoneme_predictions.json
csv_rows = []           # for per_results.csv

total_edits = 0
total_ref_tokens = 0
all_boundary_errors=[]

for _, row in tqdm(df.iterrows(), total=len(df)):

    audio_path = os.path.join(audio_dir, row['audio_filename'])

    try:
        pred_phonemes, pred_alignments = \
            get_phoneme_alignments(model, processor, audio_path)

        # Normalize
        ref_raw = row.get("reference_phonemes", "")
        ref_norm = normalize_reference(ref_raw)
        hyp_norm = normalize_hypothesis(pred_phonemes)

        distance, ref_len = compute_per(ref_norm, hyp_norm)

        if ref_len > 0:
            file_per = distance / ref_len
            total_edits += distance
            total_ref_tokens += ref_len
        else:
            file_per = 0

        # ---- JSON output (keep full alignment info)
        """json_results.append({
            "audio_filename": row["audio_filename"],
            "predicted_phonemes": pred_phonemes,
            "alignments": pred_alignments
        })

        # ---- CSV output (PER + phonemes)
        csv_rows.append({
            "audio_filename": row["audio_filename"],
            "file_per": file_per,
            "reference_phonemes_raw": ref_raw,
            "predicted_phonemes_raw": pred_phonemes,
            "reference_phonemes_norm": ref_norm,
            "predicted_phonemes_norm": hyp_norm
        })"""
        textgrid_path = os.path.join(
            textgrid_dir,
            row["audio_filename"].replace(".wav", "-Pro.TextGrid")
        )

        ref_alignments = get_reference_alignments(textgrid_path)
        print("Ref alignments:", len(ref_alignments))
        print("Pred alignments:", len(pred_alignments))
        ref_offset = ref_alignments[0]["start"]
        #hyp_offset = pred_alignments[0]["start"]
    
        ref_intervals = [{
                "phoneme": item["phoneme"],
                "start": item["start"] - ref_offset,
                "end": item["end"] - ref_offset} for item in ref_alignments]
        boundary_errors = match_alignments(ref_alignments, pred_alignments)

        all_boundary_errors.extend(boundary_errors)


    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        continue

corpus_per = total_edits / total_ref_tokens if total_ref_tokens > 0 else 0



if len(all_boundary_errors) > 0:
    errors = np.array(all_boundary_errors)

    mean_boundary_error = np.mean(errors)
    median_boundary_error = np.median(errors)

    within_20ms = np.mean(errors <= 0.02) * 100
    within_50ms = np.mean(errors <= 0.05) * 100
else:
    mean_boundary_error = 0
    median_boundary_error = 0
    within_20ms = 0
    within_50ms = 0
print("\n===== FINAL RESULTS =====")
print(f"Corpus PER: {corpus_per * 100:.2f}%")
print(f"Mean boundary error: {mean_boundary_error * 1000:.2f} ms")
print(f"Median boundary error: {median_boundary_error * 1000:.2f} ms")
print(f"% within 20ms: {within_20ms:.2f}%")
print(f"% within 50ms: {within_50ms:.2f}%")

# ---- Save JSON (phoneme predictions + timestamps)
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print(f"Phoneme predictions saved to {json_output_path}")

# ---- Save CSV (PER analysis)
pd.DataFrame(csv_rows).to_csv(csv_output_path, index=False)

print(f"PER results saved to {csv_output_path}")



  2%|████▊                                                                                                                                                                                                                                                                               | 1/57 [00:02<01:59,  2.13s/it]

Ref alignments: 313
Pred alignments: 385


  4%|█████████▋                                                                                                                                                                                                                                                                          | 2/57 [00:19<10:08, 11.07s/it]

Ref alignments: 2629
Pred alignments: 3054


  5%|██████████████▌                                                                                                                                                                                                                                                                     | 3/57 [00:23<07:02,  7.82s/it]

...

 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏    | 56/57 [10:41<00:03,  3.14s/it]

Ref alignments: 161
Pred alignments: 208


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [10:42<00:00, 11.27s/it]

Ref alignments: 201
Pred alignments: 251

===== FINAL RESULTS =====
Corpus PER: 23.23%
Mean boundary error: 16874.16 ms
Median boundary error: 776.91 ms
% within 20ms: 3.53%
% within 50ms: 6.93%


NameError: name 'json_output_path' is not defined

In [ ]:


if len(all_boundary_errors) > 0:
    errors = np.array(all_boundary_errors)

    mean_boundary_error = np.mean(errors)
    median_boundary_error = np.median(errors)

    within_20ms = np.mean(errors <= 0.02) * 100
    within_50ms = np.mean(errors <= 0.05) * 100
else:
    mean_boundary_error = 0
    median_boundary_error = 0
    within_20ms = 0
    within_50ms = 0
print("\n===== FINAL RESULTS =====")
print(f"Corpus PER: {corpus_per * 100:.2f}%")
print(f"Mean boundary error: {mean_boundary_error * 1000:.2f} ms")
print(f"Median boundary error: {median_boundary_error * 1000:.2f} ms")
print(f"% within 20ms: {within_20ms:.2f}%")
print(f"% within 50ms: {within_50ms:.2f}%")

# ---- Save JSON (phoneme predictions + timestamps)
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print(f"Phoneme predictions saved to {json_output_path}")

# ---- Save CSV (PER analysis)
pd.DataFrame(csv_rows).to_csv(csv_output_path, index=False)

print(f"PER results saved to {csv_output_path}")


In [14]:
def split_ipa_string(text, multi_char_phonemes):
    tokens = []
    i = 0
    
    # Sort longest first
    multi_char_phonemes = sorted(multi_char_phonemes, key=len, reverse=True)
    
    while i < len(text):
        match = None
        
        # Try multi-character phonemes first
        for ph in multi_char_phonemes:
            if text.startswith(ph, i):
                match = ph
                break
        
        if match:
            tokens.append(match)
            i += len(match)
        else:
            # Otherwise single character phoneme
            tokens.append(text[i])
            i += 1
            
    return tokens


In [15]:
df1=pd.read_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/per_results.csv")


In [6]:
#SAMBA to IPA
ref_mapping = {
    "a": "a",
    "b": "b",
    "c": "k",
    "d": "d",
    "Z": "ʒ",
    "e": "e",
    "f": "f",
    "i": "i",
    "j": "j",
    "k": "k",
    "l": "l",
    "n": "n",
    "o": "o",
    "p": "p",
    "s": "s",
    "t": "t",
    "u": "u",
    "v": "v",
    "w": "w",
    "y": "y",
    "z": "z",
    "2": "ø",
    "9": "œ",
    "N": "ŋ",
    "@": "ə",
    "E": "ɛ",
    "O": "ɔ",
    "R": "ʁ",
    "r": "ʁ"
    "S": "ʃ",
    "J": "ɲ",
    "H": "ɥ",
    "g": "ɡ",
    "g":"ɡ",

    # Nasals
    "a~": "ɑ̃",
    "o~": "ɔ̃",
    "e~": "ɛ̃",
    "9~": "ɛ̃",
}
#IPA to SAMBA
hyp_mapping = {

    # Multilingual vowel projection
    "ɪ": "i",
    "ʊ": "u",
    "ʌ": "ɔ",
    "ɜ": "ə",
    "ɨ": "i",

    # French mergers
    "ɑ": "a",
    "ɒ": "ɔ",

    # Rhotic variants
    "ɣ": "ʁ",
    "ɹ": "ʁ",
    "ɾ": "ʁ",

    # Palatal lateral
    "ʎ": "l",

    # Palatalized consonants
    "mʲ": "m",

    # Affricates collapse
    "tʃ": "ʃ",
    "ts": "s",

    # Greek / foreign consonants
    "β": "b",
    "θ": "s",

    # NASAL VOWELS → match reference inventory
    "ã": "ɑ̃",
    "ẽ": "ɛ̃",
    "ĩ": "ɛ̃",
    "õ": "ɔ̃",
    "ũ": "ɔ̃",
    "ỹ": "ɛ̃",
}



def normalize_reference(ref_string):

    ref_string = unicodedata.normalize("NFC", ref_string)

    # Replace longest first (nasals first)
    for k in sorted(ref_mapping.keys(), key=len, reverse=True):
        ref_string = ref_string.replace(k, ref_mapping[k])

    # Remove noise
    for noise in ["_", "sil", "spn", "%", "?", "0", "="]:
        ref_string = ref_string.replace(noise, "")

    # Remove combining tilde if any left
    ref_string = ref_string.replace("̃", "")

    ref_string = ref_string.replace(" ", "")

    return ref_string


def normalize_hypothesis(decoded_string):

    import unicodedata

    hyp = unicodedata.normalize("NFC", decoded_string)
    hyp = hyp.replace(" ", "")

    for k in sorted(hyp_mapping.keys(), key=len, reverse=True):
        hyp = hyp.replace(k, hyp_mapping[k])

    # remove length markers
    hyp = hyp.replace("ː", "")

    # remove stray combining tilde
    hyp = hyp.replace("̃", "")

    return hyp


SyntaxError: invalid syntax (2730454275.py, line 32)

In [46]:
df.columns

Index(['audio_filename', 'file_per', 'reference_phonemes_raw',
       'predicted_phonemes_raw', 'reference_phonemes_norm',
       'predicted_phonemes_norm'],
      dtype='object')

In [65]:
import pandas as pd

df = pd.read_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/per_results.csv")

def extract_chars(raw):
    chars = set()
    for seq in raw:
        for char in seq:
            chars.add(char)
    return chars
df["ref_normalized"] = df["reference_phonemes_raw"].apply(normalize_reference)
df["pred_normalized"] = df["predicted_phonemes_raw"].apply(normalize_hypothesis)
ref_chars = extract_chars(df["ref_normalized"])
pred_chars = extract_chars(df["pred_normalized"])

print("Ref chars:", sorted(ref_chars))
print("Pred chars:", sorted(pred_chars))

print("Only in ref:", sorted(ref_chars - pred_chars))
print("Only in pred:", sorted(pred_chars - ref_chars))


Ref chars: ['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'ɑ', 'ɔ', 'ə', 'ɛ', 'ɡ', 'ɥ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Pred chars: ['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'ɑ', 'ɔ', 'ə', 'ɛ', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Only in ref: ['ɥ']
Only in pred: []


In [16]:
MULTI_CHAR_PHONEMES = [
    "ɑ̃", "ɛ̃", "ɔ̃", "œ̃"
]

In [17]:
df1.columns


Index(['audio_filename', 'file_per', 'reference_phonemes_raw',
       'predicted_phonemes_raw', 'reference_phonemes_norm',
       'predicted_phonemes_norm'],
      dtype='object')

In [19]:
df1["phoneme_tokens"] = df1["predicted_phonemes_norm"].apply(
    lambda x: split_ipa_string(x, MULTI_CHAR_PHONEMES)
)


In [6]:
import unicodedata

with open("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/output_w2vrecognition/mfa_corpus/Rhap-D2006.lab", "r", encoding="utf-8") as f:
    text = f.read()

for i, ch in enumerate(text):
    print(i, repr(ch), unicodedata.name(ch))


0 'm' LATIN SMALL LETTER M
1 ' ' SPACE
2 ' ' SPACE
3 ' ' SPACE
4 'ə' LATIN SMALL LETTER SCHWA
5 ' ' SPACE
6 ' ' SPACE
7 ' ' SPACE
8 's' LATIN SMALL LETTER S
9 ' ' SPACE
10 ' ' SPACE
11 ' ' SPACE
12 'j' LATIN SMALL LETTER J
13 ' ' SPACE
14 ' ' SPACE
15 ' ' SPACE
16 'ø' LATIN SMALL LETTER O WITH STROKE
17 ' ' SPACE
18 ' ' SPACE
19 ' ' SPACE
20 'l' LATIN SMALL LETTER L
21 ' ' SPACE
22 ' ' SPACE
23 ' ' SPACE
24 'ə' LATIN SMALL LETTER SCHWA
25 ' ' SPACE
26 ' ' SPACE
27 ' ' SPACE
28 'p' LATIN SMALL LETTER P
29 ' ' SPACE
30 ' ' SPACE
31 ' ' SPACE
32 'ʁ' LATIN LETTER SMALL CAPITAL INVERTED R
33 ' ' SPACE
34 ' ' SPACE
35 ' ' SPACE
36 'ə' LATIN SMALL LETTER SCHWA
37 ' ' SPACE
38 ' ' SPACE
39 ' ' SPACE
40 'm' LATIN SMALL LETTER M
41 ' ' SPACE
42 ' ' SPACE
43 ' ' SPACE
44 'j' LATIN SMALL LETTER J
45 ' ' SPACE
46 ' ' SPACE
47 ' ' SPACE
48 'e' LATIN SMALL LETTER E
49 ' ' SPACE
50 ' ' SPACE
51 ' ' SPACE
52 'm' LATIN SMALL LETTER M
53 ' ' SPACE
54 ' ' SPACE
55 ' ' SPACE
56 'i' LATIN SMALL LETTER I
57 

## extract files for mfa

In [29]:
import pandas as pd
import ast
#df1["predicted_phonemes"]=[" ".join(ast.literal_eval(i)) for i in df1["predicted_phonemes"]]
#df1 = df1.rename(columns={"audio_filename": "filename"})
#typaloc
patho ="park"
df1=pd.read_csv(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/pred_w2v_{patho}.csv")
df1["speaker_id"] =[i.split("_")[0] for i in df1["filename"]]
df1

,filename,predicted_phonemes,speaker_id
0,CCM-003733-01_L01,d ɑ̃ l a p ə t i v i l a ʒ d ə m ɔ̃ t a ɲ d ə ...,CCM-003733-01
1,CCM-003148-01_L01,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003148-01
2,CCM-003848-01_L01,ʁ ə m e ʁ a k ɔ̃ t d ɑ̃ z ɛ̃ p ə t i v i l a ʒ...,CCM-003848-01
3,CCM-003734-01_L01,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003734-01
4,CCM-003346-01_L01,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003346-01
5,CCM-003557-01_L01,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003557-01
6,CCM-001773-01_L01,d ø v a j ɑ̃ p ə t i l y t l ɛ̃ ɡ ʁ o m e l ʁ ...,CCM-001773-01
7,CCM-003130-01_L01,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003130-01


In [ ]:
#monpage
#df1=pd.read_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/pred_w2v_mon.csv")
#df1["speaker_id"] = ["_".join(i.split("_")[:3]) for i in df1["filename"]]


In [81]:
#df1["filename1"]=[i.split("-")[1].split(".")[0]+"-"+i.split("-")[0]+".wav" for i in df1["filename"]]

#df1["speaker_id"]=[i.split("-")[0] for i in df1["filename1"]]
df1 = df1[df1["speaker_id"] != "D2004"]

In [72]:
#df1["speaker_id"]=[i.split("-")[1] for i in df1["filename"]]

In [30]:
def normalize_phones(seq):
    tokens = seq.strip().split()
    new_tokens = []

    for t in tokens:
        if t == "ɑ":        # only replace standalone ɑ
            new_tokens.append("a")
        if t =='ø̃':
            new_tokens.append('ø')
        else:
            new_tokens.append(t)

    return " ".join(new_tokens)

df1["predicted_phonemes"] = df1["predicted_phonemes"].apply(normalize_phones)


In [31]:
ref_inventory = set()

for seq in df1["predicted_phonemes"].dropna():
    tokens = seq.split()
    ref_inventory.update(tokens)

print(sorted(ref_inventory))
print("Number of unique phonemes:", len(ref_inventory))

['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Number of unique phonemes: 33


In [32]:
import os
import torch
import torchaudio
import shutil

corpus_dir = f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/mfa_{patho}_w2v_ctc"
if os.path.exists(corpus_dir):
    shutil.rmtree(corpus_dir)  

In [33]:
#monpage
os.makedirs(corpus_dir)
target_sr = 16000
#wav_path = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
wav_path = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-PARK"
for idx, row in df1.iterrows():
    #audio_path = os.path.join(wav_path, row["filename"][:-4]+".wav")
    audio_path = os.path.join(wav_path, row["filename"])+".wav"
    #audio_path = os.path.join(wav_path, row["filename"][:-11]+".wav")
    tokens = row["predicted_phonemes"]
    #audio_path1 = os.path.join(wav_path, row["filename1"])
    #utt_id = os.path.splitext(os.path.basename(audio_path1))[0]
    utt_id = os.path.splitext(os.path.basename(audio_path))[0]
    #utt_id = str(row["filename"])
    speaker_id = str(row["speaker_id"])   # ← IMPORTANT

    # Create speaker folder
    speaker_dir = os.path.join(corpus_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Save wav inside speaker folder
    torchaudio.save(
        os.path.join(speaker_dir, f"{utt_id}.wav"),
        waveform,
        target_sr
    )

    # Save lab inside speaker folder
    with open(os.path.join(speaker_dir, f"{utt_id}.lab"), "w", encoding="utf-8") as f:
        f.write(tokens.strip())



/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch

In [34]:
with open("/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/output_ph_reco/phoneme_park_w2v_ctc.txt", "w", encoding="utf-8") as f:
    for ph in ref_inventory:
        f.write(f"{ph} {ph}\n")

In [16]:
#tmp=pd.read_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/logs/csv_files/rhap/rhap_sans_hesitation_ni_rep.csv")
#tmp["speaker_id"] = [i.split("-")[1].split(".")[0] for i in tmp["filename"]]

In [17]:
#tmp["filename1"]=[i.split("-")[1].split(".")[0]+"-"+i.split("-")[0]+".wav" for i in tmp["filename"]]

In [14]:
def normalization(text):
    # 1. Unicode normalize
    text = unicodedata.normalize("NFKD", text)

    # 2. Remove accents (diacritics)
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch) != "Mn"
    )

    # 3. Remove text inside square brackets
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # 4. Remove text inside parentheses
    text = re.sub(r"\([^)]*\)", " ", text)

    # 5. Remove symbols & punctuation
    text = "".join(
        " " if unicodedata.category(ch)[0] in {"S", "P"} else ch
        for ch in text
    )

    # 6. Lowercase
    text = text.lower()

    # 7. Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text.split(" ")


In [40]:
tmp  =pd.read_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/logs/transcription_files/typaloc-ctrl/wav2vec2-VAD-chunk-trans.csv")
tmp["speaker_id"] = [i.split("-")[1] for i in tmp["filename"]]
list(tmp["filename"])

['AEX-CAB000-02_L01.wav',
 'AEX-CAC000-02_L01.wav',
 'AEX-CAG000-01_L01.wav',
 'AEX-CLJ000-02_L01.wav',
 'AEX-CML000-01_L01.wav',
 'AEX-CSR000-01_L01.wav',
 'BEX-CDB000-01_L01.wav',
 'BEX-CEB000-01_L01.wav',
 'BEX-CHE000-01_L01.wav',
 'BEX-CKN000-01_L01.wav',
 'BEX-CMB000-01_L01.wav',
 'BEX-CNKN00-01_L01.wav']

In [42]:
from pathlib import Path
import os
import ast
import shutil
import unicodedata
import re

corpus_dir = Path("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/ctrl_asrw2v")
corpus_dir.mkdir(exist_ok=True)

target_sr = 16000
wav_path = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/"
for idx, row in tmp.iterrows():
    audio_path = os.path.join(wav_path, row["filename"])
    tokens = normalization(" ".join(ast.literal_eval(row["pred_trans"])))
    #audio_path1 = os.path.join(wav_path, row["filename1"])
    #utt_id = os.path.splitext(os.path.basename(audio_path1))[0]
    utt_id = os.path.splitext(os.path.basename(audio_path))[0]
    speaker_id = str(row["speaker_id"])   # ← IMPORTANT

    # Create speaker folder
    speaker_dir = os.path.join(corpus_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Save wav inside speaker folder
    torchaudio.save(
        os.path.join(speaker_dir, f"{utt_id}.wav"),
        waveform,
        target_sr
    )

    # Save lab inside speaker folder
    with open(os.path.join(speaker_dir, f"{utt_id}.lab"), "w", encoding="utf-8") as f:
        f.write(" ".join(tokens))

In [20]:
tokens

['']